In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [10]:
###### import warnings
warnings.filterwarnings("ignore")

import copy
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import DataLoader, TensorDataset


# ============================================================
# CONFIG  (v8 — threshold-based + tuned)
# ============================================================

DATA_PATH  = "/kaggle/input/datasets/arjunmahesh09999/new-masterdata/MASTERDATA.csv"
OUTPUT_DIR = Path("cnn_gru_v9_binary_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_SEED     = 7
ROWS_PER_MIN    = 30

BURN_IN_ROWS    = 200
TRIM_FIRST_ROWS = 200
TRIM_LAST_ROWS  = 200

VAL_TEST_TRIM_FIRST = 400
VAL_TEST_TRIM_LAST  = 200

WINDOW_ROWS = 80   
STRIDE_ROWS = 30

TEST_PATIENT_FRAC = 0.10
VAL_PATIENT_FRAC  = 0.10

TARGET_COL        = "future_label"
BINARY_TARGET_COL = "binary_future_label"
PATIENT_COL       = "patient_id"

BATCH_SIZE   = 256
MAX_EPOCHS   = 140

# ── v8 tuning rationale ──────────────────────────────────────────────────────

LR           = 8e-6       
WARMUP_EPOCHS= 6        
WEIGHT_DECAY = 1e-5
PATIENCE     = 20         
GRAD_CLIP    = 5.0

FOCAL_GAMMA  = 2.0        
LABEL_SMOOTH = 0.01

CLASS0_BOOST = 1.0       

SWA_START_FRAC = 0.55    
SWA_LR         = LR * 0.5

# ── Jitter augmentation ───────────────────────────────────────────────────────
JITTER_STD  = 0.015
JITTER_PROB = 0.5


THRESHOLD_STRATEGIES = ["youden", "f1", "sens_85", "sens_90"]
DEFAULT_STRATEGY     = "youden"       

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# ============================================================
# FEATURES  (unchanged from v7)
# ============================================================
VITAL_LIMITS = {
    "dbp":               (20, 150),
    "mbp":               (30, 200),
    "sbp":               (50, 280),
    "heart_rate":        (20, 220),
    "spo2":              (50, 100),
    "etco2":             (5,  80),
    "pulse_pressure":    (5,  150),
    "resp_rate_smoothed":(4,  60),
}
RAW_VITALS = list(VITAL_LIMITS.keys())

SLOPE_PREFS  = ["slope_5m_", "slope_7m_", "slope_15m_"]
SLOPE_VITALS = ["spo2","heart_rate","resp_rate_smoothed","sbp","dbp","mbp","etco2","pulse_pressure"]
SLOPE_COLS   = [f"{p}{v}" for p in SLOPE_PREFS for v in SLOPE_VITALS]

COMBINED_SLOPE_COLS = [
    "slope_2m_combined_score",
    "slope_5m_combined_score",
    "slope_7m_combined_score",
    "slope_15m_combined_score",
]

EXTRA_COLS = [
    "combined_score",
    "roll_mean_15m_combined",
    "roll_min_15m_combined",
    "roll_max_15m_combined",
    "roll_std_15m_combined",
]

TARGETED_FLAG_COLS = [
    "t3_masked_shock",
    "t3_stable_deceiver",
    "t3_occult_acidosis",
]

VITAL_NORM = {
    "dbp":               (70.0,  12.5),
    "mbp":               (90.0,  15.0),
    "sbp":              (120.0,  23.75),
    "heart_rate":        (75.0,  18.75),
    "spo2":              (98.0,   2.5),
    "etco2":             (40.0,   7.5),
    "pulse_pressure":    (50.0,  13.75),
    "resp_rate_smoothed":(16.0,   5.5),
    "combined_score":           (0.0,  0.5),
    "roll_mean_15m_combined":   (0.2,  0.4),
    "roll_min_15m_combined":    (0.0,  0.4),
    "roll_max_15m_combined":    (0.3,  0.4),
    "roll_std_15m_combined":    (0.0,  0.15),
}

SLOPE_VITAL_STD = {
    "slope_5m":  0.015,
    "slope_7m":  0.012,
    "slope_15m": 0.008,
}
COMBINED_SLOPE_STD = {
    "slope_2m_combined_score": 0.005,
    "slope_5m_combined_score": 0.003,
    "slope_7m_combined_score": 0.002,
    "slope_15m_combined_score":0.001,
}


# ============================================================
# PREPROCESSING  (unchanged)
# ============================================================
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    print("* Feature engineering...")

    def _patient(g):
        g = g.copy()
        for v in RAW_VITALS:
            if v not in g.columns:
                continue
            s    = g[v].astype(float)
            prev = s.shift(1)
            jump = (prev.notna()) & ((s > 2.0*prev.abs()) | (s < 0.5*prev.abs()))
            s[jump] = np.nan
            g[v] = s.ffill()
        for v, (lo, hi) in VITAL_LIMITS.items():
            if v in g.columns:
                g.loc[(g[v] < lo) | (g[v] > hi), v] = np.nan
        return g

    df = df.groupby(PATIENT_COL, group_keys=False).apply(_patient).reset_index(drop=True)

    print("* Creating binary target column (0=Normal, 1=Critical+Emergency)...")
    df[BINARY_TARGET_COL] = df[TARGET_COL].apply(
        lambda x: 0 if x == 0 else (1 if x in [1, 2] else np.nan)
    )
    print(f"  Binary label distribution:\n{df[BINARY_TARGET_COL].value_counts(dropna=True)}")
    print(f"  Columns after engineering: {df.shape[1]}")
    return df


def trim_edges(df, first_rows=None, last_rows=None):
    if first_rows is None:
        first_rows = BURN_IN_ROWS + TRIM_FIRST_ROWS
    if last_rows is None:
        last_rows = TRIM_LAST_ROWS
    print(f"* Trimming first {first_rows} and last {last_rows} rows...")

    def _trim(g):
        if len(g) <= (first_rows + last_rows):
            return g.iloc[0:0]
        return g.iloc[first_rows : len(g) - last_rows]

    return df.groupby(PATIENT_COL, group_keys=False).apply(_trim).reset_index(drop=True)


def build_feature_cols(df):
    candidates = (
        RAW_VITALS + SLOPE_COLS + COMBINED_SLOPE_COLS +
        EXTRA_COLS + TARGETED_FLAG_COLS
    )
    present = [c for c in candidates if c in df.columns]
    print(f"* Final Feature Count: {len(present)}")
    return present


def apply_fixed_normalisation(X, feature_cols):
    X = np.nan_to_num(X.copy().astype(np.float32), nan=0.0)
    for fi, col in enumerate(feature_cols):
        if col in VITAL_NORM:
            ref, std = VITAL_NORM[col]
            X[:, :, fi] = (X[:, :, fi] - ref) / (std + 1e-8)
            continue
        matched = False
        for key, std in COMBINED_SLOPE_STD.items():
            if key in col:
                X[:, :, fi] /= (std + 1e-8)
                matched = True
                break
        if matched:
            continue
        for prefix, std in SLOPE_VITAL_STD.items():
            if prefix in col:
                X[:, :, fi] /= (std + 1e-8)
                break
    return X


# ============================================================
# SPLIT + WINDOWS  (unchanged)
# ============================================================
def split_patients(df):
    pids = df[PATIENT_COL].unique().copy()
    rng  = np.random.default_rng(RANDOM_SEED)
    rng.shuffle(pids)

    n_test     = max(1, int(len(pids) * TEST_PATIENT_FRAC))
    test_pids  = pids[:n_test]
    rem        = pids[n_test:]
    n_val      = max(1, int(len(rem) * VAL_PATIENT_FRAC))
    val_pids   = rem[:n_val]
    train_pids = rem[n_val:]

    train_df = df[df[PATIENT_COL].isin(train_pids)].reset_index(drop=True)
    val_df   = df[df[PATIENT_COL].isin(val_pids)].reset_index(drop=True)
    test_df  = df[df[PATIENT_COL].isin(test_pids)].reset_index(drop=True)

    print(f"* Splits (before trim): Train={len(train_pids)}, Val={len(val_pids)}, Test={len(test_pids)}")
    return train_df, val_df, test_df


def make_windows(df, feature_cols):
    X_list, y_list, pid_list = [], [], []
    for pid in df[PATIENT_COL].unique():
        g      = df[df[PATIENT_COL] == pid].reset_index(drop=True)
        feat   = g[feature_cols].values.astype(np.float32)
        labels = g[BINARY_TARGET_COL].values
        n      = len(g)
        start  = 0
        while start + WINDOW_ROWS <= n:
            end   = start + WINDOW_ROWS
            label = labels[end - 1]
            if not np.isnan(label):
                X_list.append(feat[start:end])
                y_list.append(int(label))
                pid_list.append(pid)
            start += STRIDE_ROWS

    if len(X_list) == 0:
        return (
            np.empty((0, WINDOW_ROWS, len(feature_cols)), dtype=np.float32),
            np.array([]), np.array([]),
        )
    return np.stack(X_list), np.array(y_list), np.array(pid_list)


# ============================================================
# FOCAL LOSS
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0, reduction="mean"):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.reduction       = reduction

    def forward(self, logits, targets):
        n_classes = logits.size(1)
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, self.label_smoothing / n_classes)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing + self.label_smoothing / n_classes)
        log_probs    = F.log_softmax(logits, dim=1)
        probs        = log_probs.exp()
        pt           = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        ce           = -(smooth_targets * log_probs).sum(dim=1)
        focal_weight = (1.0 - pt) ** self.gamma
        loss         = focal_weight * ce
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean() if self.reduction == "mean" else loss.sum()


# ============================================================
# MODEL  — v8 adds a third residual block + wider GRU
# ============================================================
class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.3):
        super().__init__()
        mid = out_ch // 3

        def _branch(k):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=k, padding=k//2, bias=False),
                nn.BatchNorm1d(mid),
                nn.GELU(),
            )

        self.b3  = _branch(3)
        self.b7  = _branch(7)
        self.b11 = _branch(11)

        self.proj = nn.Sequential(
            nn.Conv1d(3 * mid, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip = (
            nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        branches = torch.cat([self.b3(x), self.b7(x), self.b11(x)], dim=1)
        return self.proj(branches) + self.skip(x)


class ResidualCNNBlock(nn.Module):
    def __init__(self, ch, kernel_size=3, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(ch, ch, kernel_size=kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm1d(ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


class CNNGRU(nn.Module):
    """
    v8: 3 residual blocks (was 2) + GRU hidden 64→80 + wider head.
    Roughly +20% params for better temporal feature extraction.
    """
    def __init__(self, n_features, n_classes=2, dropout=0.4):
        super().__init__()
        self.ms_block = MultiScaleBlock(n_features, 96, dropout=dropout * 0.75)
        self.res1 = ResidualCNNBlock(96, kernel_size=5, dropout=dropout * 0.75)
        self.res2 = ResidualCNNBlock(96, kernel_size=3, dropout=dropout * 0.75)
        self.res3 = ResidualCNNBlock(96, kernel_size=3, dropout=dropout * 0.75)  # NEW

        self.gru = nn.GRU(
            input_size=96, hidden_size=80,          # 64→80
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )

        self.attn = nn.Sequential(
            nn.Linear(160, 80), nn.Tanh(), nn.Linear(80, 1)
        )

        self.head = nn.Sequential(
            nn.Linear(160, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        z    = self.ms_block(x)
        z    = self.res1(z)
        z    = self.res2(z)
        z    = self.res3(z)
        z    = z.transpose(1, 2)
        h, _ = self.gru(z)
        w    = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        ctx  = (h * w.unsqueeze(-1)).sum(1)
        return self.head(ctx)


# ============================================================
# DATA LOADERS
# ============================================================
class JitterDataset(torch.utils.data.Dataset):
    def __init__(self, X_t, y_t, jitter_std=0.015, jitter_prob=0.5):
        self.X   = X_t
        self.y   = y_t
        self.std = jitter_std
        self.p   = jitter_prob

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if torch.rand(1).item() < self.p:
            x = x + torch.randn_like(x) * self.std
        return x, self.y[idx]


def make_loader(X, y, shuffle, augment=False):
    Xt = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    if augment:
        dataset = JitterDataset(Xt, yt, jitter_std=JITTER_STD, jitter_prob=JITTER_PROB)
    else:
        dataset = TensorDataset(Xt, yt)
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=2, pin_memory=(DEVICE.type == "cuda"),
    )


# ============================================================
# TEMPERATURE SCALING
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature

    def calibrate(self, logits_np, labels_np):
        logits_t = torch.tensor(logits_np, dtype=torch.float32).to(DEVICE)
        labels_t = torch.tensor(labels_np, dtype=torch.long).to(DEVICE)
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def _eval():
            optimizer.zero_grad()
            loss = F.cross_entropy(logits_t / self.temperature, labels_t)
            loss.backward()
            return loss

        optimizer.step(_eval)
        T = self.temperature.item()
        print(f"  Learned temperature T = {T:.4f}  (>1 = softer, <1 = sharper)")
        return T


# ============================================================
# THRESHOLD OPTIMISATION  ← NEW CORE ADDITION
# ============================================================

def find_optimal_threshold(y_true, proba_pos, strategy="youden"):
    """
    Find decision threshold on a calibration split (val set).

    Strategies
    ----------
    youden      : argmax(TPR − FPR)  — balanced sensitivity+specificity
    f1          : argmax F1(At-Risk) — overall performance on positive class
    sens_85     : highest threshold s.t. sensitivity >= 0.85  (best specificity)
    sens_90     : highest threshold s.t. sensitivity >= 0.90  (best specificity)

    Returns
    -------
    threshold : float in [0, 1]
    achieved  : dict with {sensitivity, specificity, f1, precision} at threshold
    """
    fpr, tpr, thresholds = roc_curve(y_true, proba_pos, pos_label=1)
    # roc_curve returns decreasing thresholds; tpr/fpr match
    # thresholds[0] corresponds to the "all positive" extreme

    if strategy == "youden":
        j     = tpr - fpr
        idx   = int(np.argmax(j))
        best_t = float(thresholds[idx])

    elif strategy == "f1":
        best_t, best_f1 = 0.5, 0.0
        for t in np.linspace(0.05, 0.95, 181):
            preds = (proba_pos >= t).astype(int)
            f = f1_score(y_true, preds, pos_label=1, zero_division=0)
            if f > best_f1:
                best_f1, best_t = f, float(t)

    elif strategy in ("sens_85", "sens_90"):
        target_sens = 0.85 if strategy == "sens_85" else 0.90
       
        best_t = float(thresholds[-1])   
        for t_val, tpr_val in zip(thresholds, tpr):
            if tpr_val >= target_sens:
                best_t = float(t_val)
                break

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # Compute achieved metrics at best_t
    preds = (proba_pos >= best_t).astype(int)
    cm    = confusion_matrix(y_true, preds, labels=[0, 1])
    TN, FP = cm[0, 0], cm[0, 1]
    FN, TP = cm[1, 0], cm[1, 1]
    achieved = {
        "sensitivity": TP / (TP + FN) if (TP + FN) > 0 else 0.0,
        "specificity": TN / (TN + FP) if (TN + FP) > 0 else 0.0,
        "f1":          f1_score(y_true, preds, pos_label=1, zero_division=0),
        "precision":   precision_score(y_true, preds, pos_label=1, zero_division=0),
    }
    return best_t, achieved


def fit_all_thresholds(y_val, proba_val):
    """
    Run all threshold strategies on val set; return dict of results.
    """
    results = {}
    print("\n── Threshold Optimisation (on Val Set) ──")
    print(f"  {'Strategy':<14} {'Threshold':>10} {'Sensitivity':>12} "
          f"{'Specificity':>12} {'F1(AR)':>8} {'Precision':>10}")
    print("  " + "-" * 70)
    for strategy in THRESHOLD_STRATEGIES:
        t, achieved = find_optimal_threshold(y_val, proba_val[:, 1], strategy=strategy)
        results[strategy] = {"threshold": t, **achieved}
        print(f"  {strategy:<14} {t:>10.4f} {achieved['sensitivity']:>12.4f} "
              f"{achieved['specificity']:>12.4f} {achieved['f1']:>8.4f} "
              f"{achieved['precision']:>10.4f}")
    return results


# ============================================================
# COLLECT LOGITS
# ============================================================
@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    all_logits, all_y = [], []
    for X_b, y_b in loader:
        X_b = X_b.to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
        all_logits.append(logits.float().cpu().numpy())
        all_y.append(y_b.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)


# ============================================================
# TRAIN / EVAL
# ============================================================
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = correct = n = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
            loss   = criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, temp_scaler=None):
    model.eval()
    total_loss = correct = n = 0
    all_probs, all_y = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(X_b)
            if temp_scaler is not None:
                logits = temp_scaler(logits)
            loss = criterion(logits, y_b)
        probs       = torch.softmax(logits.float(), dim=1)
        total_loss += loss.item() * len(y_b)
        correct    += (logits.argmax(1) == y_b).sum().item()
        n          += len(y_b)
        all_probs.append(probs.cpu().numpy())
        all_y.append(y_b.cpu().numpy())
    proba = np.concatenate(all_probs)
    y     = np.concatenate(all_y)
    try:
        auroc   = roc_auc_score(y, proba[:, 1])
        preds   = proba.argmax(axis=1)
        rec1    = recall_score(y, preds, pos_label=1, average="binary", zero_division=0)
        monitor = 0.6 * auroc + 0.4 * rec1
    except Exception:
        auroc = rec1 = monitor = float("nan")
    return total_loss / n, correct / n, auroc, rec1, monitor, y, proba


# ============================================================
# FIT — with SWA + linear warmup  (v8)
# ============================================================
def fit_model(model, tr_loader, va_loader, class_weights):
    focal_weight    = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion       = FocalLoss(weight=focal_weight, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTH)
    plain_criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Linear warmup for WARMUP_EPOCHS, then cosine decay
    def warmup_cosine(ep):
        if ep < WARMUP_EPOCHS:
            return (ep + 1) / WARMUP_EPOCHS          
        progress = (ep - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        return 0.05 + 0.95 * 0.5 * (1 + np.cos(np.pi * progress))   

    scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)
    amp_scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    swa_model     = AveragedModel(model)
    swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR)
    swa_started   = False

    best_monitor = -1.0
    best_state   = copy.deepcopy(model.state_dict())
    patience     = 0
    history      = {"train_loss":[], "val_loss":[], "val_auroc":[], "val_rec1":[], "val_monitor":[]}

    print(f"\n{'Ep':>4} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} "
          f"{'Val Acc':>9} {'AUROC':>7} {'Rec1':>6} {'Monitor':>8}")
    print("-" * 75)

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(model, tr_loader, optimizer, criterion, amp_scaler)
        va_loss, va_acc, va_auroc, va_rec1, va_monitor, _, _ = eval_epoch(
            model, va_loader, plain_criterion
        )

        if ep >= int(MAX_EPOCHS * SWA_START_FRAC):
            if not swa_started:
                print(f"\n* SWA started at epoch {ep}")
                swa_started = True
            swa_model.update_parameters(model)
            swa_scheduler.step()
        else:
            scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["val_auroc"].append(va_auroc)
        history["val_rec1"].append(va_rec1)
        history["val_monitor"].append(va_monitor)

        mark     = " *" if va_monitor > best_monitor else ""
        swa_mark = " [SWA]" if swa_started else ""
        print(f"{ep:4d} {tr_loss:11.4f} {tr_acc:10.4f} {va_loss:10.4f} "
              f"{va_acc:9.4f} {va_auroc:7.4f} {va_rec1:6.4f} {va_monitor:8.4f}{mark}{swa_mark}")

        if va_monitor > best_monitor:
            best_monitor = va_monitor
            best_state   = copy.deepcopy(model.state_dict())
            patience     = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f"\n* Early stopping at epoch {ep}")
                break

    if swa_started:
        print("\n* Updating BatchNorm for SWA model...")
        update_bn(tr_loader, swa_model, device=DEVICE)
        _, _, swa_auroc, swa_rec1, swa_monitor, _, _ = eval_epoch(swa_model, va_loader, plain_criterion)
        print(f"  SWA  monitor={swa_monitor:.4f} (AUROC={swa_auroc:.4f} Rec1={swa_rec1:.4f})")
        print(f"  Best monitor={best_monitor:.4f}")
        if swa_monitor > best_monitor:
            print("  → Using SWA weights.")
            model.load_state_dict(swa_model.module.state_dict())
            best_monitor = swa_monitor
        else:
            print("  → Keeping best checkpoint weights.")
            model.load_state_dict(best_state)
    else:
        model.load_state_dict(best_state)

    return model, history, best_monitor


# ============================================================
# REPORTING — threshold-based  (v8 replaces argmax)
# ============================================================
def decision_report(y_true, proba, label, threshold=0.5, strategy_name="argmax"):
    """
    Full classification report using a calibrated probability threshold.

    At threshold=0.5 this is identical to argmax.
    At threshold<0.5 sensitivity (At-Risk recall) increases at cost of specificity.
    """
    pos_proba = proba[:, 1]
    y_pred    = (pos_proba >= threshold).astype(int)  

    print(f"\n{'='*60}")
    print(f"  {label}  [strategy={strategy_name}, t={threshold:.4f}]")
    print(f"{'='*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=["Normal (0)", "At-Risk (1+2)"],
        digits=4,
    ))

    auroc = roc_auc_score(y_true, pos_proba)
    auprc = average_precision_score(y_true, pos_proba)
    bal   = balanced_accuracy_score(y_true, y_pred)
    f1_pos  = f1_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0)
    rec_pos = recall_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0)
    pre_pos = precision_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0)

    cm  = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP = cm[0, 0], cm[0, 1]
    FN, TP = cm[1, 0], cm[1, 1]
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    print(f"  AUROC        = {auroc:.4f}")
    print(f"  AUPRC        = {auprc:.4f}")
    print(f"  Balanced Acc = {bal:.4f}")
    print(f"  Precision(1) = {pre_pos:.4f}")
    print(f"  Recall(1)    = {rec_pos:.4f}")
    print(f"  F1(1)        = {f1_pos:.4f}")
    print(f"\n  Confusion Matrix (rows=True, cols=Predicted):")
    print(f"               Pred Normal  Pred At-Risk")
    print(f"  True Normal :    {TN:6d}       {FP:6d}")
    print(f"  True At-Risk:    {FN:6d}       {TP:6d}")
    print(f"\n  Sensitivity (Recall At-Risk) = {sensitivity:.4f}")
    print(f"  Specificity (Recall Normal)  = {specificity:.4f}")
    print(f"  TP={TP}  FP={FP}  TN={TN}  FN={FN}")

    return {
        "auroc": auroc, "auprc": auprc, "bal": bal,
        "f1": f1_pos, "recall": rec_pos, "precision": pre_pos,
        "sensitivity": sensitivity, "specificity": specificity,
        "y_true": y_true, "y_pred": y_pred, "proba": proba, "cm": cm,
        "threshold": threshold,
    }


def report_all_strategies(y_true, proba_te, threshold_results):
    """Print test results for every strategy threshold."""
    print("\n" + "="*65)
    print("  TEST RESULTS — All Threshold Strategies")
    print("="*65)
    print(f"  {'Strategy':<14} {'Thresh':>7} {'Sens':>7} {'Spec':>7} "
          f"{'F1':>7} {'Prec':>7} {'BalAcc':>8}")
    print("  " + "-"*60)

    pos_proba = proba_te[:, 1]
    all_res   = {}
    for strategy, info in threshold_results.items():
        t     = info["threshold"]
        preds = (pos_proba >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        TN, FP = cm[0, 0], cm[0, 1]
        FN, TP = cm[1, 0], cm[1, 1]
        sens  = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec  = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        f1    = f1_score(y_true, preds, pos_label=1, zero_division=0)
        prec  = precision_score(y_true, preds, pos_label=1, zero_division=0)
        bal   = balanced_accuracy_score(y_true, preds)
        star  = " ←" if strategy == DEFAULT_STRATEGY else ""
        print(f"  {strategy:<14} {t:>7.4f} {sens:>7.4f} {spec:>7.4f} "
              f"{f1:>7.4f} {prec:>7.4f} {bal:>8.4f}{star}")
        all_res[strategy] = {
            "threshold": t, "sensitivity": sens, "specificity": spec,
            "f1": f1, "precision": prec, "bal": bal,
            "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        }
    print(f"\n  ← = DEFAULT_STRATEGY for deployment ({DEFAULT_STRATEGY})")
    return all_res


# ============================================================
# PLOTTING
# ============================================================
def plot_training(history, out_path):
    fig, ax = plt.subplots(figsize=(10, 5), facecolor="white")
    ax.plot(history["train_loss"], label="train_loss", color="C0")
    ax.plot(history["val_loss"],   label="val_loss",   color="C1")
    ax.set_ylabel("Loss"); ax.set_xlabel("Epoch")
    ax2 = ax.twinx()
    ax2.plot(history["val_auroc"],   color="C2", linestyle="-",  label="val_auroc")
    ax2.plot(history["val_rec1"],    color="C3", linestyle="--", label="val_rec1")
    ax2.plot(history["val_monitor"], color="C4", linestyle=":",  label="val_monitor (0.6A+0.4R)")
    ax2.set_ylabel("Metric")
    ax.set_title("Training curves — v9 Binary")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
    fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_confusion(cm, out_path, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(5, 4), facecolor="white")
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Normal", "At-Risk"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Normal", "At-Risk"])
    fig.colorbar(im, ax=ax)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_roc_curve(y_true, pos_proba, threshold_results, out_path):
    fpr, tpr, _ = roc_curve(y_true, pos_proba)
    auroc = roc_auc_score(y_true, pos_proba)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    ax.plot(fpr, tpr, label=f"AUROC = {auroc:.4f}", lw=2)
    ax.plot([0, 1], [0, 1], "k--", lw=1)

    # Mark each threshold strategy on the ROC curve
    colors = {"youden": "tab:red", "f1": "tab:green",
               "sens_85": "tab:orange", "sens_90": "tab:purple"}
    for strategy, info in threshold_results.items():
        t    = info["threshold"]
        sens = info["sensitivity"]
        spec = info["specificity"]
        ax.scatter(1 - spec, sens, marker="o", s=70, color=colors.get(strategy, "k"),
                   zorder=5, label=f"{strategy} (t={t:.2f})")

    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Binary (Normal vs At-Risk)")
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_pr_curve(y_true, pos_proba, out_path):
    pre, rec, _ = precision_recall_curve(y_true, pos_proba)
    auprc = average_precision_score(y_true, pos_proba)
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    ax.plot(rec, pre, label=f"AUPRC = {auprc:.4f}", lw=2)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — Binary")
    ax.legend(); fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


def plot_threshold_sweep(y_true, pos_proba, out_path):
    """Plot Sensitivity, Specificity, F1 vs threshold (val set)."""
    thresholds = np.linspace(0.05, 0.95, 181)
    sens_list, spec_list, f1_list = [], [], []
    for t in thresholds:
        preds = (pos_proba >= t).astype(int)
        cm    = confusion_matrix(y_true, preds, labels=[0, 1])
        TN, FP = cm[0, 0], cm[0, 1]
        FN, TP = cm[1, 0], cm[1, 1]
        sens_list.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        spec_list.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
        f1_list.append(f1_score(y_true, preds, pos_label=1, zero_division=0))

    fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
    ax.plot(thresholds, sens_list, label="Sensitivity", color="tab:blue")
    ax.plot(thresholds, spec_list, label="Specificity", color="tab:orange")
    ax.plot(thresholds, f1_list,   label="F1 (At-Risk)", color="tab:green")
    ax.axhline(0.85, color="gray", linestyle="--", lw=1, label="Sens=0.85 target")
    ax.axhline(0.90, color="gray", linestyle=":",  lw=1, label="Sens=0.90 target")
    ax.set_xlabel("Decision Threshold"); ax.set_ylabel("Metric Value")
    ax.set_title("Threshold Sweep (Val Set) — v8 Binary")
    ax.legend(); fig.tight_layout(); fig.savefig(out_path, dpi=140); plt.close(fig)


# ============================================================
# MAIN
# ============================================================
def main():
    print("CNN-GRU v9 Binary | Fixed Threshold Search + Youden Default")
    print("Target: binary_future_label  (0=Normal, 1=Critical+Emergency)\n")

    df = pd.read_csv(DATA_PATH)
    print(f"* Initial Data Shape: {df.shape}")

    df = preprocess(df)
    df = df.dropna(subset=[TARGET_COL, BINARY_TARGET_COL]).reset_index(drop=True)
    df[TARGET_COL]        = df[TARGET_COL].astype(int)
    df[BINARY_TARGET_COL] = df[BINARY_TARGET_COL].astype(int)

    print(f"\n* Binary label distribution (full dataset):")
    print(f"  {df[BINARY_TARGET_COL].value_counts().to_dict()}")

    FEATURES = build_feature_cols(df)

    train_df, val_df, test_df = split_patients(df)

    print("\n* Applying per-split trimming...")
    train_df = trim_edges(train_df)
    val_df   = trim_edges(val_df,  VAL_TEST_TRIM_FIRST, VAL_TEST_TRIM_LAST)
    test_df  = trim_edges(test_df, VAL_TEST_TRIM_FIRST, VAL_TEST_TRIM_LAST)

    print("\n* Building windows (binary labels)...")
    X_tr, y_tr, _ = make_windows(train_df, FEATURES)
    X_va, y_va, _ = make_windows(val_df,   FEATURES)
    X_te, y_te, _ = make_windows(test_df,  FEATURES)

    print(f"  Train: {X_tr.shape}  labels: {np.bincount(y_tr) if len(y_tr) else []}")
    print(f"  Val  : {X_va.shape}  labels: {np.bincount(y_va) if len(y_va) else []}")
    print(f"  Test : {X_te.shape}  labels: {np.bincount(y_te) if len(y_te) else []}")

    X_tr = apply_fixed_normalisation(X_tr, FEATURES)
    X_va = apply_fixed_normalisation(X_va, FEATURES)
    X_te = apply_fixed_normalisation(X_te, FEATURES)
    print("* Fixed clinical normalisation applied")

    n0, n1   = np.bincount(y_tr)
    classes  = np.array([0, 1])
    weights  = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    cw       = dict(zip(classes.tolist(), weights.tolist()))
    cw[0]    = cw[0] * CLASS0_BOOST
    class_weights = np.array([cw[0], cw[1]], dtype=np.float32)
    print(f"* Window counts : Normal={n0}, At-Risk={n1}  (ratio={n1/n0:.2f}x)")
    print(f"* Class Weights : Normal={class_weights[0]:.4f}, At-Risk={class_weights[1]:.4f}"
          f"  (Normal/At-Risk ratio = {class_weights[0]/class_weights[1]:.2f}x)")

    tr_loader = make_loader(X_tr, y_tr, shuffle=True,  augment=True)
    va_loader = make_loader(X_va, y_va, shuffle=False, augment=False)
    te_loader = make_loader(X_te, y_te, shuffle=False, augment=False)

    model = CNNGRU(n_features=len(FEATURES), n_classes=2, dropout=0.4).to(DEVICE)
    print(f"* Model params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    print("\n--- Starting Training ---")
    model, history, best_val_monitor = fit_model(model, tr_loader, va_loader, class_weights)

    # ── Temperature calibration on val set ───────────────────────────────────
    print("\n* Calibrating temperature on val set...")
    val_logits, val_labels = collect_logits(model, va_loader)
    temp_scaler = TemperatureScaler().to(DEVICE)
    T = temp_scaler.calibrate(val_logits, val_labels)

    plain_criterion = nn.CrossEntropyLoss()

    # ── Get calibrated probabilities for val and test ─────────────────────────
    _, _, _, _, _, y_va_true, proba_va = eval_epoch(
        model, va_loader, plain_criterion, temp_scaler=temp_scaler
    )
    _, _, _, _, _, y_te_true, proba_te = eval_epoch(
        model, te_loader, plain_criterion, temp_scaler=temp_scaler
    )

    # ── Find optimal thresholds on val set ───────────────────────────────────
    threshold_results = fit_all_thresholds(y_va_true, proba_va)

    # ── Report with default strategy on test ─────────────────────────────────
    default_t  = threshold_results[DEFAULT_STRATEGY]["threshold"]
    print(f"\n── Test Set Results (strategy={DEFAULT_STRATEGY}, t={default_t:.4f}) ──")
    res = decision_report(
        y_te_true, proba_te,
        label="TEST — Binary (Normal vs At-Risk)",
        threshold=default_t,
        strategy_name=DEFAULT_STRATEGY,
    )

    # ── All strategies on test ────────────────────────────────────────────────
    all_test_res = report_all_strategies(y_te_true, proba_te, threshold_results)

    # ── Final summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  FINAL SUMMARY  (default strategy: {DEFAULT_STRATEGY})")
    print(f"{'='*60}")
    print(f"  Best Val Monitor (0.6*AUROC+0.4*Rec1): {best_val_monitor:.4f}")
    print(f"  Test AUROC        : {res['auroc']:.4f}")
    print(f"  Test AUPRC        : {res['auprc']:.4f}")
    print(f"  Test Balanced Acc : {res['bal']:.4f}")
    print(f"  Test F1 (At-Risk)  : {res['f1']:.4f}")
    print(f"  Test Precision(1)  : {res['precision']:.4f}")
    print(f"  Test Recall(1)     : {res['recall']:.4f}")
    print(f"  Test Sensitivity   : {res['sensitivity']:.4f}")
    print(f"  Test Specificity   : {res['specificity']:.4f}")
    print(f"  Temperature T      : {T:.4f}")
    print(f"  Decision Threshold : {default_t:.4f}  (strategy={DEFAULT_STRATEGY})")

    # ── Plots ─────────────────────────────────────────────────────────────────
    plot_training(history,              OUTPUT_DIR / "training_curves.png")
    plot_confusion(res["cm"],           OUTPUT_DIR / "confusion_binary.png",
                   f"Confusion Matrix — Binary [{DEFAULT_STRATEGY}]")
    plot_roc_curve(res["y_true"],       res["proba"][:, 1],
                   all_test_res,        OUTPUT_DIR / "roc_curve.png")
    plot_pr_curve(res["y_true"],        res["proba"][:, 1],
                  OUTPUT_DIR / "pr_curve.png")
    plot_threshold_sweep(y_va_true,     proba_va[:, 1],
                         OUTPUT_DIR / "threshold_sweep.png")

    # ── Save artefacts ────────────────────────────────────────────────────────
    joblib.dump(
        {
            "state_dict":           model.state_dict(),
            "temp_scaler_state":    temp_scaler.state_dict(),
            "temperature":          T,
            "features":             FEATURES,
            "class_weights":        class_weights,
            "history":              history,
            "best_val_monitor":     best_val_monitor,
            "task":                 "binary",
            "label_map":            {0: "Normal", 1: "At-Risk (Critical+Emergency)"},
            # Threshold artefacts — one per strategy
            "threshold_results":    threshold_results,   # from val set
            "all_test_results":     all_test_res,        # test metrics per strategy
            "default_strategy":     DEFAULT_STRATEGY,
            "default_threshold":    default_t,
        },
        OUTPUT_DIR / "cnn_gru_v9_binary_model.pkl",
    )
    np.save(OUTPUT_DIR / "feature_cols.npy", np.array(FEATURES))
    print(f"\nArtifacts saved to {OUTPUT_DIR}")
    return model, history, res, threshold_results


if __name__ == "__main__":
    model, history, test_result, threshold_results = main()

Device: cuda
CNN-GRU v9 Binary | Fixed Threshold Search + Youden Default
Target: binary_future_label  (0=Normal, 1=Critical+Emergency)

* Initial Data Shape: (2378857, 99)
* Feature engineering...
* Creating binary target column (0=Normal, 1=Critical+Emergency)...
  Binary label distribution:
binary_future_label
1    1511550
0     867307
Name: count, dtype: int64
  Columns after engineering: 100

* Binary label distribution (full dataset):
  {1: 1511550, 0: 867307}
* Final Feature Count: 44
* Splits (before trim): Train=314, Val=34, Test=38

* Applying per-split trimming...
* Trimming first 400 and last 200 rows...
* Trimming first 400 and last 200 rows...
* Trimming first 400 and last 200 rows...

* Building windows (binary labels)...
  Train: (57300, 80, 44)  labels: [23028 34272]
  Val  : (6630, 80, 44)  labels: [2323 4307]
  Test : (6876, 80, 44)  labels: [2109 4767]
* Fixed clinical normalisation applied
* Window counts : Normal=23028, At-Risk=34272  (ratio=1.49x)
* Class Weights 